# Solar Eclipse Shadow on Earth

This notebook computes the next solar eclipse (Sun occulted by the Moon) and visualizes the Moon's shadow sweeping across the Earth's surface over time.

## Setup

In [ ]:
import numpy as np
import plotly.graph_objs as go

from ostk.mathematics.geometry.d3.object import Point as Point3d
from ostk.mathematics.geometry.d3.object import Ray as Ray3d
from ostk.mathematics.geometry.d3.object import Cone

from ostk.physics import Environment
from ostk.physics.time import Scale
from ostk.physics.time import Instant
from ostk.physics.time import Duration
from ostk.physics.time import Interval
from ostk.physics.time import DateTime
from ostk.physics.coordinate import Frame
from ostk.physics.coordinate import Position
from ostk.physics.coordinate import Velocity
from ostk.physics.coordinate.spherical import LLA
from ostk.physics.environment.object import Geometry
from ostk.physics.environment.object.celestial import Earth
from ostk.physics.environment.object.celestial import Sun
from ostk.physics.environment.object.celestial import Moon

from ostk.astrodynamics.solver import TemporalConditionSolver

---

## Environment

In [ ]:
environment = Environment.default()

earth = environment.access_celestial_object_with_name("Earth")
sun = environment.access_celestial_object_with_name("Sun")
moon = environment.access_celestial_object_with_name("Moon")

## Find the Next Solar Eclipse

We detect a solar eclipse by checking when the **line segment between the Sun and Moon centers** intersects with the **Earth geometry**.
When this intersection exists, the Moon's shadow is falling somewhere on Earth's surface.

We use the `TemporalConditionSolver` to find the precise time intervals.

In [ ]:
search_interval = Interval.closed(
    Instant.parse("2026-08-01 00:00:00", Scale.UTC),
    Instant.parse("2026-09-01 00:00:00", Scale.UTC),
)

solver = TemporalConditionSolver(
    time_step=Duration.minutes(30.0),
    tolerance=Duration.seconds(10.0),
)

In [ ]:
def sun_moon_line_intersects_earth(instant: Instant) -> bool:
    sun_pos = sun.get_position_in(Frame.GCRF(), instant).get_coordinates()
    moon_pos = moon.get_position_in(Frame.GCRF(), instant).get_coordinates()

    # Moon must be between Sun and Earth (closer to Sun than Earth is)
    d_sun_moon = np.linalg.norm(moon_pos - sun_pos)
    d_sun_earth = np.linalg.norm(sun_pos)  # Earth is at GCRF origin
    if d_sun_moon > d_sun_earth:
        return False

    direction = (moon_pos - sun_pos) / d_sun_moon
    ray = Ray3d(Point3d(*sun_pos), direction)
    ray_geometry = Geometry(ray, Frame.GCRF())
    earth_geometry = earth.get_geometry_in(Frame.GCRF(), instant)
    return ray_geometry.intersects(earth_geometry)


eclipse_windows = solver.solve(
    condition=sun_moon_line_intersects_earth,
    interval=search_interval,
)

print(f"Found {len(eclipse_windows)} solar eclipse window(s):")
for i, window in enumerate(eclipse_windows):
    print(f"  Eclipse {i + 1}: {window}")

## Compute the Shadow Geometry

For each timestep during the eclipse, we:
1. Get the **Sun** and **Moon** geometries (spheres) in ITRF using `get_geometry_in`
2. Construct a **shadow cone** — apex near the Moon, pointing away from the Sun, with half-angle derived from the Sun-Moon geometry
3. Intersect the shadow cone with the **Earth** geometry to obtain the shadow footprint
4. Convert the intersection to geodetic coordinates (latitude / longitude)

In [ ]:
# Use the first eclipse window found
eclipse_interval = eclipse_windows[0]

# Expand slightly to capture partial phases (penumbra extends beyond the central line)
eclipse_start = eclipse_interval.get_start() - Duration.minutes(30.0)
eclipse_end = eclipse_interval.get_end() + Duration.minutes(30.0)
eclipse_window = Interval.closed(eclipse_start, eclipse_end)

print(f"Eclipse window: {eclipse_start.to_string()} → {eclipse_end.to_string()}")

In [ ]:
R_SUN = 696340e3  # [m]
R_MOON = 1737.4e3  # [m]


def make_circle_latlng(center_lat_deg, center_lon_deg, radius_km, n_points=64):
    """Approximate a circle on Earth's surface (great-circle offsets)."""
    R_EARTH_KM = 6371.0
    angles = np.linspace(0, 2 * np.pi, n_points, endpoint=True)
    angular_dist = radius_km / R_EARTH_KM

    lat0 = np.radians(center_lat_deg)
    lon0 = np.radians(center_lon_deg)

    lats = np.arcsin(
        np.sin(lat0) * np.cos(angular_dist)
        + np.cos(lat0) * np.sin(angular_dist) * np.cos(angles)
    )
    lons = lon0 + np.arctan2(
        np.sin(angles) * np.sin(angular_dist) * np.cos(lat0),
        np.cos(angular_dist) - np.sin(lat0) * np.sin(lats),
    )

    return np.degrees(lats), np.degrees(lons)


def point_gcrf_to_lla(coords_gcrf, instant):
    """Convert a GCRF position vector to (lat_deg, lon_deg) via ITRF."""
    pos_gcrf = Position.meters(coords_gcrf.tolist(), Frame.GCRF())
    coords_itrf = pos_gcrf.in_frame(Frame.ITRF(), instant).get_coordinates()
    lla = LLA.cartesian(
        coords_itrf,
        earth.get_equatorial_radius(),
        earth.get_flattening(),
    )
    return float(lla.get_latitude().in_degrees()), float(lla.get_longitude().in_degrees())


def compute_shadow_at_instant(instant):
    """
    Compute penumbra and umbra shadow footprints on Earth at a given instant.

    Returns dict with center LLA, penumbra/umbra radii and circle coordinates,
    or None if the shadow misses Earth.
    """
    sun_pos = sun.get_position_in(Frame.GCRF(), instant).get_coordinates()
    moon_pos = moon.get_position_in(Frame.GCRF(), instant).get_coordinates()

    d_sun_moon = np.linalg.norm(sun_pos - moon_pos)

    # Shadow axis: unit vector from Sun through Moon toward Earth
    shadow_axis = (moon_pos - sun_pos) / d_sun_moon

    # Sub-shadow point: where the shadow axis intersects Earth's surface
    # Parametric: P = moon_pos + t * shadow_axis, find t where |P| = R_earth
    R_EARTH = 6371e3
    a_coeff = 1.0
    b_coeff = 2.0 * np.dot(moon_pos, shadow_axis)
    c_coeff = np.dot(moon_pos, moon_pos) - R_EARTH**2

    discriminant = b_coeff**2 - 4.0 * a_coeff * c_coeff
    if discriminant < 0:
        return None

    t = (-b_coeff - np.sqrt(discriminant)) / (2.0 * a_coeff)
    shadow_center_gcrf = moon_pos + t * shadow_axis

    center_lat, center_lon = point_gcrf_to_lla(shadow_center_gcrf, instant)

    # Penumbra half-angle: external tangent (Sun edge to opposite Moon edge)
    penumbra_half_angle = np.arctan((R_SUN + R_MOON) / d_sun_moon)
    dist_moon_to_ground = np.abs(t)
    penumbra_radius_km = (dist_moon_to_ground * np.tan(penumbra_half_angle)) / 1e3

    # Umbra half-angle: internal tangent (Sun edge to same-side Moon edge)
    umbra_half_angle = np.arctan((R_SUN - R_MOON) / d_sun_moon)
    umbra_apex_dist = R_MOON / np.tan(umbra_half_angle)
    umbra_ground_dist = dist_moon_to_ground - umbra_apex_dist
    umbra_radius_km = np.abs(umbra_ground_dist * np.tan(umbra_half_angle)) / 1e3

    # Cap penumbra at a reasonable display size
    penumbra_radius_km = min(penumbra_radius_km, 4000.0)

    penumbra_lats, penumbra_lons = make_circle_latlng(
        center_lat, center_lon, penumbra_radius_km
    )
    umbra_lats, umbra_lons = make_circle_latlng(
        center_lat, center_lon, umbra_radius_km
    )

    return {
        "center_lat": center_lat,
        "center_lon": center_lon,
        "penumbra_radius_km": penumbra_radius_km,
        "umbra_radius_km": umbra_radius_km,
        "penumbra_lats": penumbra_lats,
        "penumbra_lons": penumbra_lons,
        "umbra_lats": umbra_lats,
        "umbra_lons": umbra_lons,
    }

In [ ]:
step = Duration.minutes(5.0)
instants = eclipse_window.generate_grid(step)

shadow_data = []
for instant in instants:
    result = compute_shadow_at_instant(instant)
    if result is not None:
        result["instant"] = instant
        result["time_str"] = instant.to_string()
        shadow_data.append(result)

print(f"Computed shadow at {len(shadow_data)} timesteps")
if shadow_data:
    print(f"  First: {shadow_data[0]['time_str']}")
    print(f"  Last:  {shadow_data[-1]['time_str']}")
    print(
        f"  Umbra radius range: {min(s['umbra_radius_km'] for s in shadow_data):.0f}"
        f" – {max(s['umbra_radius_km'] for s in shadow_data):.0f} km"
    )
    print(
        f"  Penumbra radius range: {min(s['penumbra_radius_km'] for s in shadow_data):.0f}"
        f" – {max(s['penumbra_radius_km'] for s in shadow_data):.0f} km"
    )

---

## Dynamic Shadow Map

An animated map showing the **penumbra** (partial shadow, light gray) and **umbra** (total shadow, dark) sweeping across Earth's surface.
The red line traces the ground track of the shadow center.

In [ ]:
# Ground track of the shadow center
track_lats = [s["center_lat"] for s in shadow_data]
track_lons = [s["center_lon"] for s in shadow_data]

# Build animation frames
frames = []
slider_steps = []

for i, s in enumerate(shadow_data):
    frame_data = [
        # Penumbra boundary
        go.Scattergeo(
            lat=s["penumbra_lats"].tolist(),
            lon=s["penumbra_lons"].tolist(),
            mode="lines",
            line=dict(width=1, color="rgba(100,100,100,0.5)"),
            fill="toself",
            fillcolor="rgba(100,100,100,0.15)",
            name="Penumbra",
            showlegend=(i == 0),
        ),
        # Umbra boundary
        go.Scattergeo(
            lat=s["umbra_lats"].tolist(),
            lon=s["umbra_lons"].tolist(),
            mode="lines",
            line=dict(width=2, color="rgba(30,30,30,0.8)"),
            fill="toself",
            fillcolor="rgba(30,30,30,0.4)",
            name="Umbra",
            showlegend=(i == 0),
        ),
        # Shadow center marker
        go.Scattergeo(
            lat=[s["center_lat"]],
            lon=[s["center_lon"]],
            mode="markers",
            marker=dict(size=8, color="black", symbol="circle"),
            name="Shadow Center",
            showlegend=(i == 0),
        ),
        # Ground track up to current time
        go.Scattergeo(
            lat=track_lats[: i + 1],
            lon=track_lons[: i + 1],
            mode="lines",
            line=dict(width=2, color="red"),
            name="Ground Track",
            showlegend=(i == 0),
        ),
    ]

    frames.append(go.Frame(data=frame_data, name=str(i)))
    slider_steps.append(
        dict(
            args=[[str(i)], dict(frame=dict(duration=100, redraw=True), mode="immediate")],
            label=s["time_str"][-15:-5],  # show HH:MM:SS portion
            method="animate",
        )
    )

# Initial frame
fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text="Solar Eclipse Shadow — Moon's Shadow on Earth"),
        geo=dict(
            showland=True,
            landcolor="rgb(243, 243, 243)",
            countrycolor="rgb(204, 204, 204)",
            coastlinecolor="rgb(150, 150, 150)",
            showocean=True,
            oceancolor="rgb(200, 220, 240)",
            showcountries=True,
            projection_type="natural earth",
            showframe=False,
        ),
        height=700,
        width=1100,
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                x=0.05,
                y=0.0,
                xanchor="left",
                yanchor="top",
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=200, redraw=True),
                                fromcurrent=True,
                                mode="immediate",
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                steps=slider_steps,
                x=0.05,
                len=0.9,
                xanchor="left",
                y=-0.05,
                currentvalue=dict(prefix="Time: ", visible=True),
                transition=dict(duration=100),
            )
        ],
    ),
)

fig.show()

---

## Eclipse Summary

In [ ]:
# Find the timestep with the smallest umbra (closest to totality center)
min_umbra_idx = min(range(len(shadow_data)), key=lambda i: shadow_data[i]["umbra_radius_km"])
max_eclipse = shadow_data[min_umbra_idx]

print("=" * 60)
print("  SOLAR ECLIPSE SUMMARY")
print("=" * 60)
print(f"  Eclipse interval:  {eclipse_interval.get_start().to_string()}")
print(f"                  →  {eclipse_interval.get_end().to_string()}")
print(f"  Maximum eclipse:   {max_eclipse['time_str']}")
print(f"    Center:          ({max_eclipse['center_lat']:.2f}°, {max_eclipse['center_lon']:.2f}°)")
print(f"    Umbra radius:    {max_eclipse['umbra_radius_km']:.1f} km")
print(f"    Penumbra radius: {max_eclipse['penumbra_radius_km']:.0f} km")
print(f"  Ground track:")
print(f"    Start:           ({shadow_data[0]['center_lat']:.1f}°, {shadow_data[0]['center_lon']:.1f}°)")
print(f"    End:             ({shadow_data[-1]['center_lat']:.1f}°, {shadow_data[-1]['center_lon']:.1f}°)")
print("=" * 60)

---

## Shadow Footprint via OSTk Geometry Intersection

As an alternative to the analytical approach above, we can use OSTk's geometry engine directly.
We construct a `Cone` representing the shadow (umbra or penumbra), then call `intersection_with` on the Earth geometry to get the precise footprint.

In [ ]:
from ostk.mathematics.geometry import Angle as MathAngle

# Pick the instant of maximum eclipse
max_instant = max_eclipse["instant"]

# Get celestial body geometries in GCRF
earth_geometry = earth.get_geometry_in(Frame.GCRF(), max_instant)
sun_geometry = sun.get_geometry_in(Frame.GCRF(), max_instant)
moon_geometry = moon.get_geometry_in(Frame.GCRF(), max_instant)

# Get positions in GCRF
sun_pos = sun.get_position_in(Frame.GCRF(), max_instant).get_coordinates()
moon_pos = moon.get_position_in(Frame.GCRF(), max_instant).get_coordinates()

d_sun_moon = np.linalg.norm(sun_pos - moon_pos)
shadow_dir = (moon_pos - sun_pos) / d_sun_moon

# Penumbra cone: apex at Sun center, half-angle = arctan((R_sun + R_moon) / d_sun_moon)
penumbra_half_angle_rad = np.arctan((R_SUN + R_MOON) / d_sun_moon)

penumbra_cone = Cone(
    Point3d(*sun_pos),
    shadow_dir.tolist(),
    MathAngle.radians(penumbra_half_angle_rad),
)

penumbra_geometry = Geometry(penumbra_cone, Frame.GCRF())

# Intersect with Earth geometry
penumbra_footprint = penumbra_geometry.intersection_with(earth_geometry)

if penumbra_footprint.is_defined():
    print("Penumbra footprint intersection found via OSTk geometry engine!")

    composite = penumbra_footprint.access_composite()
    n_objects = composite.get_object_count()
    print(f"  Intersection composite has {n_objects} object(s)")
    
    footprint_llas = []
    for idx in range(n_objects):
        obj = composite.access_object_at(idx)
        if obj.is_line_string():
            line_string = obj.as_line_string()
            for pt_idx in range(line_string.get_point_count()):
                pt = line_string.access_point_at(pt_idx)
                coords = np.array([pt.x(), pt.y(), pt.z()])
                lat, lon = point_gcrf_to_lla(coords, max_instant)
                footprint_llas.append((lat, lon))

    if footprint_llas:
        print(f"  Extracted {len(footprint_llas)} footprint points")
        fp_lats = [p[0] for p in footprint_llas]
        fp_lons = [p[1] for p in footprint_llas]

        fig2 = go.Figure(
            go.Scattergeo(
                lat=fp_lats,
                lon=fp_lons,
                mode="markers+lines",
                marker=dict(size=3, color="purple"),
                line=dict(width=1, color="purple"),
                name="Penumbra footprint (OSTk intersection)",
            )
        )
        fig2.update_layout(
            title="Penumbra Footprint via OSTk Geometry Intersection",
            geo=dict(
                showland=True,
                showcountries=True,
                projection_type="natural earth",
            ),
            height=500,
            width=900,
        )
        fig2.show()
else:
    print("No intersection found (shadow may miss Earth at this instant)")

---